# 0. Import

In [64]:
!pip install wandb -q

In [65]:
import json
import wandb
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
import json
import numpy as np
from pathlib import Path


In [66]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the secret token safely
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Log into WandB
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

# 1. Making the Modality

In [67]:
def load_skeleton(trial_path):

    prediction_dir = Path(trial_path) / "predictions"

    json_files = sorted(prediction_dir.glob("*.json"))

    frames = []

    for json_file in json_files:

        with open(json_file, "r") as f:
            data = json.load(f)

        # One person per frame based on our inspection
        person = data[0]

        keypoints = np.asarray(
            person["keypoints"],
            dtype=np.float32
        )

        frames.append(keypoints)

    if len(frames) == 0:
        return None

    return np.stack(frames)

## 1. Building the training index

In [68]:
def temporal_resample(x, target_frames=64):

    T = x.shape[0]

    if T == target_frames:
        return x.astype(np.float32)

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_frames
    )

    output = np.empty(
        (target_frames, x.shape[1], x.shape[2]),
        dtype=np.float32
    )

    for joint in range(x.shape[1]):
        for coord in range(x.shape[2]):

            output[:, joint, coord] = np.interp(
                new_indices,
                old_indices,
                x[:, joint, coord]
            )

    return output

In [69]:
def normalize_skeleton(x):
    """
    x: (T, 17, 3)

    Makes skeleton coordinates root-relative.
    Joint 0 is treated as the root.
    """

    x = x.copy()

    # Root-relative coordinates
    root = x[:, 0:1, :]
    x = x - root

    return x.astype(np.float32)

def temporal_resample(x, target_frames=64):
    """
    Resample the complete sequence to exactly target_frames.
    """

    T = x.shape[0]

    if T == target_frames:
        return x.astype(np.float32)

    if T == 1:
        return np.repeat(
            x,
            target_frames,
            axis=0
        ).astype(np.float32)

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_frames
    )

    output = np.empty(
        (target_frames, x.shape[1], x.shape[2]),
        dtype=np.float32
    )

    for joint in range(x.shape[1]):

        for coord in range(x.shape[2]):

            output[:, joint, coord] = np.interp(
                new_indices,
                old_indices,
                x[:, joint, coord]
            )

    return output.astype(np.float32)

def get_bone_vectors(self, x):
    """
    x: (T, 17, 3)

    Returns:
        bone_vectors: (T, 17, 3)
    """

    # COCO-17 skeleton hierarchy
    #
    # 0  nose
    # 1  left eye
    # 2  right eye
    # 3  left ear
    # 4  right ear
    # 5  left shoulder
    # 6  right shoulder
    # 7  left elbow
    # 8  right elbow
    # 9  left wrist
    # 10 right wrist
    # 11 left hip
    # 12 right hip
    # 13 left knee
    # 14 right knee
    # 15 left ankle
    # 16 right ankle

    parents = [
        -1,  # nose
         0,  # left eye
         0,  # right eye
         1,  # left ear
         2,  # right ear
        11,  # left shoulder -> left hip
        12,  # right shoulder -> right hip
         5,  # left elbow
         6,  # right elbow
         7,  # left wrist
         8,  # right wrist
        -1,  # left hip
        -1,  # right hip
        11,  # left knee
        12,  # right knee
        13,  # left ankle
        14   # right ankle
    ]

    bone_vectors = np.zeros_like(x)

    for joint, parent in enumerate(parents):

        if parent != -1:

            bone_vectors[:, joint, :] = (
                x[:, joint, :] -
                x[:, parent, :]
            )

    return bone_vectors

## 3. Dataset Class

In [70]:
class SkeletonDataset(Dataset):
    def __init__(self, df, sequence_length=64):
        self.df = df.reset_index(drop=True)
        self.sequence_length = sequence_length

    def load_skeleton(self, path):
        prediction_dir = Path(path) / "predictions"
        json_files = sorted(prediction_dir.glob("*.json"))

        keypoint_frames = []
        confidence_frames = []

        for json_file in json_files:
            with open(json_file, "r") as f:
                data = json.load(f)

            if len(data) == 0:
                continue

            person = data[0]

            # -------------------------
            # Keypoints
            # -------------------------
            keypoints = np.asarray(
                person["keypoints"],
                dtype=np.float32
            )

            # Expected: (17, 3)
            if keypoints.shape != (17, 3):
                continue

            # -------------------------
            # Confidence scores
            # -------------------------
            scores = np.asarray(
                person["keypoint_scores"],
                dtype=np.float32
            ).reshape(-1)

            # Expected: 17 scores
            if scores.shape[0] != 17:
                continue

            scores = np.clip(scores, 0.0, 1.0)

            keypoint_frames.append(keypoints)
            confidence_frames.append(scores)

        if len(keypoint_frames) == 0:
            return None, None

        keypoints = np.stack(keypoint_frames)       # (T,17,3)
        scores = np.stack(confidence_frames)        # (T,17)

        return keypoints, scores

    # --------------------------------------------------
    # Pelvis-centered + scale normalization
    # --------------------------------------------------
    def normalize_skeleton(self, x):
        x = x.copy()

        # COCO-17
        # left hip  = 11
        # right hip = 12
        # left shoulder  = 5
        # right shoulder = 6

        pelvis = (
            x[:, 11:12, :] +
            x[:, 12:13, :]
        ) / 2.0

        x = x - pelvis

        shoulder_center = (
            x[:, 5:6, :] +
            x[:, 6:7, :]
        ) / 2.0

        scale = np.linalg.norm(
            shoulder_center,
            axis=2,
            keepdims=True
        )

        scale = np.maximum(scale, 1e-6)

        x = x / scale

        return x

    # --------------------------------------------------
    # Bone vectors
    # --------------------------------------------------
    def get_bone_vectors(self, x):

        parents = [
            -1, 0, 0, 1, 2,
            11, 12,
            5, 6,
            7, 8,
            -1, -1,
            11, 12,
            13, 14
        ]

        bones = np.zeros_like(x)

        for j, p in enumerate(parents):
            if p >= 0:
                bones[:, j, :] = x[:, j, :] - x[:, p, :]

        return bones

    # --------------------------------------------------
    # Temporal resampling
    # --------------------------------------------------
    def temporal_resample(self, x):

        T = x.shape[0]

        if T == self.sequence_length:
            return x.astype(np.float32)

        if T == 1:
            return np.repeat(
                x,
                self.sequence_length,
                axis=0
            ).astype(np.float32)

        old_indices = np.linspace(0, T - 1, T)
        new_indices = np.linspace(
            0,
            T - 1,
            self.sequence_length
        )

        output = np.empty(
            (
                self.sequence_length,
                x.shape[1],
                x.shape[2]
            ),
            dtype=np.float32
        )

        for joint in range(x.shape[1]):
            for coord in range(x.shape[2]):
                output[:, joint, coord] = np.interp(
                    new_indices,
                    old_indices,
                    x[:, joint, coord]
                )

        return output

    # --------------------------------------------------
    # Dataset
    # --------------------------------------------------
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton, confidence = self.load_skeleton(
            row["skeleton"]
        )

        # Missing skeleton
        if skeleton is None:

            skeleton = np.zeros(
                (self.sequence_length, 17, 3),
                dtype=np.float32
            )

            confidence = np.zeros(
                (self.sequence_length, 17),
                dtype=np.float32
            )

        # -------------------------
        # Normalize coordinates
        # -------------------------
        skeleton = self.normalize_skeleton(skeleton)

        # -------------------------
        # Velocity
        # -------------------------
        velocity = np.diff(
            skeleton,
            axis=0,
            prepend=skeleton[0:1]
        )

        # -------------------------
        # Acceleration
        # -------------------------
        acceleration = np.diff(
            velocity,
            axis=0,
            prepend=velocity[0:1]
        )

        # -------------------------
        # Bone vectors
        # -------------------------
        bones = self.get_bone_vectors(skeleton)

        # -------------------------
        # Add confidence
        #
        # XYZ        = 3
        # velocity   = 3
        # acceleration = 3
        # bones      = 3
        # confidence = 1
        #
        # TOTAL = 13 features/joint
        # -------------------------


        features = np.concatenate(
            [
                skeleton,
                velocity,
                acceleration,
                bones,
            ],
            axis=2
        )

        # (T, 17, 13)

        features = self.temporal_resample(features)

        # Final shape:
        # (64, 17, 13)

        X = torch.tensor(
            features,
            dtype=torch.float32
        )

        y = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return X, y

In [71]:
from pathlib import Path
import pandas as pd

TRAIN_DATA = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data"
)

MODALITIES = [
    "Skeleton",
    "Depth_Color",
    "IR",
    "Thermal",
    "IMU",
    "Radar",
]

# ---------------------------------------------------------
# Build trial index from Skeleton
# ---------------------------------------------------------

records = []

skeleton_root = TRAIN_DATA / "Skeleton"

for action_dir in sorted(skeleton_root.iterdir()):

    if not action_dir.is_dir():
        continue

    action_name = action_dir.name
    label = int(action_name.split("_")[0])

    for user_dir in sorted(action_dir.iterdir()):

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for trial_dir in sorted(user_dir.iterdir()):

            if not trial_dir.is_dir():
                continue

            records.append({
                "action": action_name,
                "label": label,
                "user": user,
                "trial": trial_dir.name,
                "path": str(trial_dir),
            })


train_df = pd.DataFrame(records)

print("Training trials:", len(train_df))
print("Classes:", train_df["label"].nunique())
print("Users:", train_df["user"].nunique())

train_df.head()

Training trials: 2931
Classes: 40
Users: 18


,action,label,user,trial,path
0,0_Wash_face,0,user16,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
1,0_Wash_face,0,user16,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...
2,0_Wash_face,0,user16,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...
3,0_Wash_face,0,user18,7-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
4,0_Wash_face,0,user18,7-1-2,/kaggle/input/datasets/samasiayushman/small-mo...


In [72]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df,
        train_df["label"],
        groups=train_df["user"]
    )
)

train_split = train_df.iloc[train_idx].reset_index(drop=True)
val_split = train_df.iloc[val_idx].reset_index(drop=True)

print("Train:", len(train_split))
print("Validation:", len(val_split))

print("Train users:")
print(sorted(train_split["user"].unique()))

print("\nValidation users:")
print(sorted(val_split["user"].unique()))

Train: 2238
Validation: 693
Train users:
['user17', 'user18', 'user19', 'user20', 'user21', 'user23', 'user24', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']

Validation users:
['user1', 'user16', 'user2', 'user22']


In [73]:
train_dataset = SkeletonDataset(
    train_split.assign(skeleton=train_split["path"]),
    sequence_length=64
)

val_dataset = SkeletonDataset(
    val_split.assign(skeleton=val_split["path"]),
    sequence_length=64
)

print("Train dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))

Train dataset: 2238
Val dataset: 693


In [74]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [75]:
X, y = next(iter(train_loader))

print("Train batch X:", X.shape)
print("Train batch y:", y.shape)

X, y = next(iter(val_loader))

print("Val batch X:", X.shape)
print("Val batch y:", y.shape)

Train batch X: torch.Size([32, 64, 17, 12])
Train batch y: torch.Size([32])
Val batch X: torch.Size([32, 64, 17, 12])
Val batch y: torch.Size([32])


In [76]:
dataset =  SkeletonDataset(
    train_split.assign(skeleton=train_split["path"]),
    sequence_length=64
)


X, y = dataset[0]

print("X shape:", X.shape)
print("y:", y)
print("dtype:", X.dtype)

print("Min:", X.min().item())
print("Max:", X.max().item())
print("Mean:", X.mean().item())
print("Std:", X.std().item())

X shape: torch.Size([64, 17, 12])
y: tensor(0)
dtype: torch.float32
Min: -1.2548960447311401
Max: 1.66280198097229
Mean: -0.0242230873554945
Std: 0.29696953296661377


---

In [77]:
print("Root joint mean:", X[:, 0, :].abs().mean().item())

print( "Root joint max:", X[:, 0, :].abs().max().item())

Root joint mean: 0.10289439558982849
Root joint max: 0.6045839190483093


# A. Baseline Model

In [78]:
import torch
import torch.nn as nn


class BiLSTMAttention(nn.Module):

    def __init__(
        self,
        input_size=102,
        hidden_size=128,
        num_layers=2,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()

        # ======================================================
        # BiLSTM
        # ======================================================

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # BiLSTM output = 128 * 2 = 256
        feature_size = hidden_size * 2

        # ======================================================
        # TEMPORAL ATTENTION
        # ======================================================

        self.attention = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # ======================================================
        # CLASSIFIER
        # ======================================================

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),

            nn.Linear(
                feature_size,
                128
            ),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(
                128,
                num_classes
            )
        )

    def extract_features(self, x):

        # ======================================================
        # Input
        # ======================================================
        # Expected:
        # (B, T, J, F)
        # Example:
        # (B, 64, 17, 6)
        # ======================================================

        B, T, J, F = x.shape

        # ======================================================
        # Flatten joints + coordinates
        # ======================================================

        x = x.reshape(
            B,
            T,
            J * F
        )

        # (B, 64, 102)

        # ======================================================
        # BiLSTM
        # ======================================================

        output, _ = self.lstm(x)

        # (B, 64, 256)

        # ======================================================
        # Temporal Attention
        # ======================================================

        scores = self.attention(output)

        # (B, 64, 1)

        weights = torch.softmax(
            scores,
            dim=1
        )

        # ======================================================
        # Weighted Temporal Representation
        # ======================================================

        context = torch.sum(
            output * weights,
            dim=1
        )

        # (B, 256)

        return context

    def forward(self, x):

        # ======================================================
        # Extract 256-D embedding
        # ======================================================

        embedding = self.extract_features(x)

        # ======================================================
        # Classification
        # ======================================================

        logits = self.classifier(
            embedding
        )

        return logits, embedding

In [79]:
class S6TemporalAttentionModel(nn.Module):

    def __init__(
        self,
        input_size=204,
        projection_size=128,
        hidden_size=128,
        num_layers=2,
        num_heads=4,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()

        # ======================================================
        # INPUT PROJECTION
        # ======================================================

        self.input_projection = nn.Sequential(

            nn.Linear(
                input_size,
                projection_size
            ),

            nn.LayerNorm(
                projection_size
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            )
        )

        # ======================================================
        # BiLSTM
        # ======================================================

        self.lstm = nn.LSTM(

            input_size=projection_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True,

            dropout=(
                dropout
                if num_layers > 1
                else 0
            ),

            bidirectional=True
        )

        feature_size = hidden_size * 2

        # ======================================================
        # MULTI-HEAD TEMPORAL ATTENTION
        # ======================================================

        self.temporal_attention = nn.MultiheadAttention(

            embed_dim=feature_size,

            num_heads=num_heads,

            dropout=dropout,

            batch_first=True
        )

        # ======================================================
        # RESIDUAL NORMALIZATION
        # ======================================================

        self.norm = nn.LayerNorm(
            feature_size
        )

        # ======================================================
        # LEARNED FRAME ATTENTION
        # ======================================================

        self.frame_attention = nn.Sequential(

            nn.Linear(
                feature_size,
                128
            ),

            nn.Tanh(),

            nn.Linear(
                128,
                1
            )
        )

        # ======================================================
        # CLASSIFIER
        # ======================================================

        self.classifier = nn.Sequential(

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                feature_size,
                128
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                128,
                num_classes
            )
        )

    def extract_features(self, x):

        # ======================================================
        # INPUT
        # ======================================================

        B, T, J, F = x.shape

        # ======================================================
        # FLATTEN JOINTS
        # ======================================================

        x = x.reshape(
            B,
            T,
            J * F
        )

        # ======================================================
        # INPUT PROJECTION
        # ======================================================

        x = self.input_projection(x)

        # (B, T, 128)

        # ======================================================
        # BiLSTM
        # ======================================================

        x, _ = self.lstm(x)

        # (B, T, 256)

        # ======================================================
        # MULTI-HEAD SELF ATTENTION
        # ======================================================

        attended, _ = self.temporal_attention(
            x,
            x,
            x
        )

        # ======================================================
        # RESIDUAL CONNECTION + NORMALIZATION
        # ======================================================

        x = self.norm(
            x + attended
        )

        # ======================================================
        # FRAME IMPORTANCE
        # ======================================================

        scores = self.frame_attention(x)

        # (B, T, 1)

        weights = torch.softmax(
            scores,
            dim=1
        )

        # ======================================================
        # WEIGHTED TEMPORAL REPRESENTATION
        # ======================================================

        embedding = torch.sum(
            x * weights,
            dim=1
        )

        # (B, 256)

        return embedding

    def forward(self, x):

        embedding = self.extract_features(x)

        logits = self.classifier(
            embedding
        )

        return logits, embedding

In [80]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = S6TemporalAttentionModel(
    input_size=204,
    projection_size=128,
    hidden_size=128,
    num_layers=2,
    num_heads=4,
    num_classes=40,
    dropout=0.3
).to(device)

print(model)
print("Device:", device)

S6TemporalAttentionModel(
  (input_projection): Sequential(
    (0): Linear(in_features=204, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
  )
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (temporal_attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
  )
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (frame_attention): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.3, inplace=False)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=40, bias=True)
  )


In [81]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [82]:
wandb.init(
    project="CIUX",
    name="Spatial BiLSTM S6",
    config={
        "model": "BiLSTM",
        "sequence_length": 64,
        "num_keypoints": 17,
        "coordinates": 3,
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.3,
        "batch_size": 32,
        "learning_rate": 1e-3,
        "epochs": 50,
        "optimizer": "Adam",
    }
)

In [83]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data/IR"
)

rows = []

for class_dir in sorted(ROOT.iterdir()):

    if not class_dir.is_dir():
        continue

    class_name = class_dir.name

    for user_dir in class_dir.iterdir():

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for seq_dir in user_dir.iterdir():

            if not seq_dir.is_dir():
                continue

            files = sorted(seq_dir.glob("*.png"))

            if not files:
                continue

            # Extract frame numbers
            frame_numbers = []

            for f in files:
                m = re.search(r'_(\d+)\.png$', f.name)

                if m:
                    frame_numbers.append(int(m.group(1)))

            rows.append({
                "class": class_name,
                "user": user,
                "sequence": seq_dir.name,
                "path": str(seq_dir),
                "n_frames": len(files),
                "first_frame": min(frame_numbers)
                    if frame_numbers else None,
                "last_frame": max(frame_numbers)
                    if frame_numbers else None
            })

df = pd.DataFrame(rows)

print("Sequences:", len(df))
print("Columns:", df.columns.tolist())

print("\nFirst rows:")
display(df.head(10))

print("\nFrames:")
print(df["n_frames"].describe())

print("\nSequences per user:")
print(df.groupby("user").size())

print("\nSequences per class:")
print(
    df.groupby("class")
      .size()
      .sort_values()
)

# CLASS MAPPING
# =========================================================

classes = sorted([
    x.name for x in ROOT.iterdir()
    if x.is_dir()
])

class_to_idx = {
    cls: i for i, cls in enumerate(classes)
}

idx_to_class = {
    i: cls for cls, i in class_to_idx.items()
}

print("Classes:", len(classes))


# =========================================================
# FRAME DATAFRAME
# =========================================================

rows = []


Sequences: 2933
Columns: ['class', 'user', 'sequence', 'path', 'n_frames', 'first_frame', 'last_frame']

First rows:


,class,user,sequence,path,n_frames,first_frame,last_frame
0,0_Wash_face,user21,4-2-3,/kaggle/input/datasets/samasiayushman/small-mo...,9,24,32
1,0_Wash_face,user21,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...,5,44,48
2,0_Wash_face,user21,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...,10,70,79
3,0_Wash_face,user21,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...,14,75,88
4,0_Wash_face,user21,4-2-1,/kaggle/input/datasets/samasiayushman/small-mo...,7,51,57
5,0_Wash_face,user21,4-2-2,/kaggle/input/datasets/samasiayushman/small-mo...,6,31,36
6,0_Wash_face,user6,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...,34,69,102
7,0_Wash_face,user6,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...,45,127,171
8,0_Wash_face,user6,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...,47,136,182
9,0_Wash_face,user18,7-1-1,/kaggle/input/datasets/samasiayushman/small-mo...,20,62,81



Frames:
count    2933.000000
mean       29.293556
std        21.894297
min         1.000000
25%        14.000000
50%        24.000000
75%        38.000000
max       236.000000
Name: n_frames, dtype: float64

Sequences per user:
user
user1     149
user16    186
user17    166
user18    178
user19    186
user2     167
user20    159
user21    133
user22    192
user23    132
user24    159
user3     161
user4     134
user5     100
user6     201
user7     184
user8     165
user9     181
dtype: int64

Sequences per class:
class
25_Watch_TV                            12
16_Fold_clothes                        24
35_Do_lunges                           26
33_Lie_down                            33
14_Wipe_bowls                          35
28_Jog_in_place                        37
18_Write                               38
26_Play_games                          40
27_Take_a_selfie                       41
3_Take_off_clothes                     41
19_Make_a_phone_call                   43
0_Wash_face

In [84]:
import os
import json
import time
import copy
import torch
import numpy as np
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/BI_LSTM_BEST")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving BiLSTM artifacts to: {SAVE_DIR}")

BILSTM_CONFIG = {

    "model": "BiLSTM",
    "version": "BI_LSTM",

    "num_classes": len(classes),
    "classes": classes,
    "class_to_idx": class_to_idx,
    "idx_to_class": idx_to_class,

    "training": {
        "epochs": 50,
        "loss": "CrossEntropyLoss",
        "optimizer": optimizer.__class__.__name__,
        "learning_rate": optimizer.param_groups[0]["lr"],
    },

    "hardware": {
        "gpu_count": torch.cuda.device_count(),
        "gpus": [
            torch.cuda.get_device_name(i)
            for i in range(torch.cuda.device_count())
        ]
    }
}

base_model = (
    model.module
    if isinstance(model, torch.nn.DataParallel)
    else model
)

num_params = sum(
    p.numel()
    for p in base_model.parameters()
    if p.requires_grad
)

BILSTM_CONFIG["parameters"] = int(num_params)
BILSTM_CONFIG["fp32_size_mb"] = float(
    num_params * 4 / 1024**2
)

print(json.dumps(BILSTM_CONFIG, indent=2))

def save_bilstm_best_artifacts(
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    best_val_acc,
    history
):

    # --------------------------------------------------------
    # Get underlying model if DataParallel is being used
    # --------------------------------------------------------

    if isinstance(model, torch.nn.DataParallel):
        base_model = model.module
    else:
        base_model = model

    state = base_model.state_dict()

    # --------------------------------------------------------
    # 1. FULL CHECKPOINT
    # --------------------------------------------------------

    checkpoint = {

        "model_state_dict": state,

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict()
            if scheduler is not None else None,

        "scaler_state_dict":
            scaler.state_dict()
            if scaler is not None else None,

        "epoch":
            int(epoch),

        "best_val_acc":
            float(best_val_acc),

        "history":
            history,

        "config":
            BILSTM_CONFIG,

        "classes":
            classes,

        "class_to_idx":
            class_to_idx,

        "idx_to_class":
            idx_to_class
    }

    torch.save(
        checkpoint,
        SAVE_DIR / "bilstm_best_checkpoint.pth"
    )

    # --------------------------------------------------------
    # 2. CLEAN FP32 WEIGHTS
    # --------------------------------------------------------

    torch.save(
        state,
        SAVE_DIR / "bilstm_best_weights.pth"
    )

    # --------------------------------------------------------
    # 3. FP16 WEIGHTS
    # --------------------------------------------------------

    fp16_state = {}

    for key, value in state.items():

        if torch.is_floating_point(value):
            fp16_state[key] = value.half()
        else:
            fp16_state[key] = value

    torch.save(
        fp16_state,
        SAVE_DIR / "bilstm_best_fp16_weights.pth"
    )

    # --------------------------------------------------------
    # 4. METADATA
    # --------------------------------------------------------

    metadata = {

        "model": "BiLSTM",

        "version": "BI_LSTM",

        "best_epoch":
            int(epoch),

        "best_validation_accuracy":
            float(best_val_acc),

        "parameters":
            int(num_params),

        "fp32_size_mb":
            float(num_params * 4 / 1024**2),

        "classes":
            classes,

        "class_to_idx":
            class_to_idx,

        "idx_to_class":
            idx_to_class,

        "fusion_note":
            "Use the BiLSTM feature representation before the final classifier for fusion."
    }

    with open(
        SAVE_DIR / "bilstm_metadata.json",
        "w"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2
        )

    # --------------------------------------------------------
    # 5. TRAINING HISTORY
    # --------------------------------------------------------

    with open(
        SAVE_DIR / "bilstm_history.json",
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )

    print(
        f"  ✓ BEST BiLSTM artifacts saved "
        f"at epoch {epoch}"
    )

def save_bilstm_last_checkpoint(
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    best_val_acc,
    history
):

    if isinstance(model, torch.nn.DataParallel):
        base_model = model.module
    else:
        base_model = model

    checkpoint = {

        "model_state_dict":
            base_model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict()
            if scheduler is not None else None,

        "scaler_state_dict":
            scaler.state_dict()
            if scaler is not None else None,

        "epoch":
            int(epoch),

        "best_val_acc":
            float(best_val_acc),

        "history":
            history,

        "config":
            BILSTM_CONFIG
    }

    torch.save(
        checkpoint,
        SAVE_DIR / "bilstm_last_checkpoint.pth"
    )

Saving BiLSTM artifacts to: /kaggle/working/BI_LSTM_BEST
{
  "model": "BiLSTM",
  "version": "BI_LSTM",
  "num_classes": 40,
  "classes": [
    "0_Wash_face",
    "10_Stir_drinks",
    "11_Peel_fruits",
    "12_Sweep_the_floor",
    "13_Mop_the_floor",
    "14_Wipe_bowls",
    "15_Wipe_windows_and_tables",
    "16_Fold_clothes",
    "17_Tap_the_keyboard",
    "18_Write",
    "19_Make_a_phone_call",
    "1_Brush_teeth",
    "20_Check_the_time",
    "21_Read_documents",
    "22_Turn_pages",
    "23_Listen_to_music_with_headphones",
    "24_Use_a_mobile_phone",
    "25_Watch_TV",
    "26_Play_games",
    "27_Take_a_selfie",
    "28_Jog_in_place",
    "29_Do_squats",
    "2_Comb_hair",
    "30_Do_jumping_jacks",
    "31_Do_stretching_exercises",
    "32_Stand_up",
    "33_Lie_down",
    "34_Sit_down",
    "35_Do_lunges",
    "36_Walk",
    "37_Take_medicine",
    "38_Massage_oneself",
    "39_Take_body_temperature",
    "3_Take_off_clothes",
    "4_Wipe_hands",
    "5_Put_on_clothes",
    

In [89]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

In [ ]:
epochs = 60
scaler = None
best_val_accuracy = 0.0

history = []

for epoch in range(epochs):

    # ==========================================================
    # TRAIN
    # ==========================================================

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X, y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits, embedding = model(X)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        train_loss += (
            loss.item() * X.size(0)
        )

        predictions = logits.argmax(dim=1)

        train_correct += (
            predictions == y
        ).sum().item()

        train_total += y.size(0)

    train_loss /= train_total

    train_accuracy = (
        train_correct / train_total
    )


    # ==========================================================
    # VALIDATION
    # ==========================================================

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for X, y in val_loader:

            X = X.to(device)
            y = y.to(device)

            logits, embedding = model(X)

            loss = criterion(logits, y)

            val_loss += (
                loss.item() * X.size(0)
            )

            predictions = logits.argmax(dim=1)

            val_correct += (
                predictions == y
            ).sum().item()

            val_total += y.size(0)

    val_loss /= val_total

    val_accuracy = (
        val_correct / val_total
    )


    # ==========================================================
    # SCHEDULER
    # ==========================================================

    
    if scheduler is not None:
        scheduler.step(val_accuracy)


    # ==========================================================
    # HISTORY
    # ==========================================================

    history.append({

        "epoch": epoch + 1,

        "train_loss":
            float(train_loss),

        "train_accuracy":
            float(train_accuracy),

        "val_loss":
            float(val_loss),

        "val_accuracy":
            float(val_accuracy),

        "learning_rate":
            float(
                optimizer.param_groups[0]["lr"]
            )
    })


    # ==========================================================
    # SAVE BEST MODEL
    # ==========================================================

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        if isinstance(
            model,
            torch.nn.DataParallel
        ):

            base_model = model.module

        else:

            base_model = model


        # ------------------------------------------------------
        # BEST WEIGHTS
        # ------------------------------------------------------

        torch.save(
            base_model.state_dict(),
            SAVE_DIR /
            "bilstm_best_weights.pth"
        )


        # ------------------------------------------------------
        # FULL CHECKPOINT
        # ------------------------------------------------------

        checkpoint = {

            "model_state_dict":
                base_model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "epoch":
                epoch + 1,

            "best_val_accuracy":
                float(best_val_accuracy),

            "history":
                history,

            "classes":
                classes,

            "class_to_idx":
                class_to_idx,

            "idx_to_class":
                idx_to_class
        }


        if scheduler is not None:

            checkpoint[
                "scheduler_state_dict"
            ] = scheduler.state_dict()


        if scaler is not None:

            checkpoint[
                "scaler_state_dict"
            ] = scaler.state_dict()


        torch.save(
            checkpoint,
            SAVE_DIR /
            "bilstm_best_checkpoint.pth"
        )


        print(
            f"🔥 New best model: "
            f"{best_val_accuracy:.4f}"
        )


    # ==========================================================
    # PRINT
    # ==========================================================

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )


    # ==========================================================
    # W&B
    # ==========================================================

    wandb.log({

        "epoch": epoch + 1,

        "train/loss":
            train_loss,

        "train/accuracy":
            train_accuracy,

        "val/loss":
            val_loss,

        "val/accuracy":
            val_accuracy,

        "learning_rate":
            optimizer.param_groups[0]["lr"]
    })


# ==========================================================
# SAVE HISTORY
# ==========================================================

with open(
    SAVE_DIR / "bilstm_history.json",
    "w"
) as f:

    json.dump(
        history,
        f,
        indent=2
    )


print("\n" + "=" * 60)

print(
    f"BEST BiLSTM VALIDATION ACCURACY: "
    f"{best_val_accuracy:.4f}"
)

print(
    f"Artifacts saved to: {SAVE_DIR}"
)

print("=" * 60)

🔥 New best model: 0.3795
Epoch 01/50 | Train Loss: 1.7996 | Train Acc: 0.4459 | Val Loss: 2.4059 | Val Acc: 0.3795
🔥 New best model: 0.4127
Epoch 02/50 | Train Loss: 1.6729 | Train Acc: 0.4803 | Val Loss: 2.2350 | Val Acc: 0.4127
🔥 New best model: 0.4170
Epoch 03/50 | Train Loss: 1.5145 | Train Acc: 0.5286 | Val Loss: 2.1675 | Val Acc: 0.4170
🔥 New best model: 0.4589
Epoch 04/50 | Train Loss: 1.3199 | Train Acc: 0.5804 | Val Loss: 2.2338 | Val Acc: 0.4589
🔥 New best model: 0.4661
Epoch 05/50 | Train Loss: 1.2738 | Train Acc: 0.5956 | Val Loss: 2.0812 | Val Acc: 0.4661
🔥 New best model: 0.4921
Epoch 06/50 | Train Loss: 1.1842 | Train Acc: 0.6139 | Val Loss: 2.1268 | Val Acc: 0.4921
Epoch 07/50 | Train Loss: 1.0601 | Train Acc: 0.6542 | Val Loss: 2.3758 | Val Acc: 0.4560
Epoch 08/50 | Train Loss: 1.0277 | Train Acc: 0.6711 | Val Loss: 2.3980 | Val Acc: 0.4329
Epoch 09/50 | Train Loss: 0.9615 | Train Acc: 0.6774 | Val Loss: 2.3629 | Val Acc: 0.4921
Epoch 10/50 | Train Loss: 0.8597 | Train

In [ ]:
wandb.finish()

---

# Z. Inference Part

## A. Test Dataset